In [8]:
# CELL 1 — Read raw EHR access logs JSON

from pyspark.sql.functions import current_timestamp, lit

batch_id = "ehr_access_logs_batch_001"
source_file = "ehr_access_logs.json"

ehr_access_logs_df = (
    spark.read
    .json("Files/raw/ehr_access_logs.json")
    .withColumn("batch_id", lit(batch_id))
    .withColumn("source_file_name", lit(source_file))
    .withColumn("ingestion_timestamp", current_timestamp())
)

print("Rows loaded:", ehr_access_logs_df.count())
ehr_access_logs_df.printSchema()
print(ehr_access_logs_df.columns)

display(ehr_access_logs_df.limit(20))

StatementMeta(, 79b6ebbf-f764-456a-a305-652b87a8210c, 10, Finished, Available, Finished, False)

Rows loaded: 1189699
root
 |-- access_log_id: string (nullable = true)
 |-- action_performed: string (nullable = true)
 |-- bytes_exported: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- employee_role: string (nullable = true)
 |-- session_end_timestamp: string (nullable = true)
 |-- session_start_timestamp: string (nullable = true)
 |-- target_encounter_id: string (nullable = true)
 |-- target_patient_id: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- workstation_ip: string (nullable = true)
 |-- batch_id: string (nullable = false)
 |-- source_file_name: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)

['access_log_id', 'action_performed', 'bytes_exported', 'device_id', 'employee_id', 'employee_role', 'session_end_timestamp', 'session_start_timestamp', 'target_encounter_id', 'target_patient_id', 'unexpected_source_field', 'workstation_ip', 'batch_id

SynapseWidget(Synapse.DataFrame, 4e79d709-5f76-4f1c-a0af-59b126b4b82a)

In [9]:
# CELL 2 — Standardize only safe column formatting

ehr_access_logs_df = ehr_access_logs_df.toDF(
    *[
        column.strip().lower()
        for column in ehr_access_logs_df.columns
    ]
)

ehr_access_logs_df.printSchema()
print(ehr_access_logs_df.columns)

StatementMeta(, 79b6ebbf-f764-456a-a305-652b87a8210c, 11, Finished, Available, Finished, False)

root
 |-- access_log_id: string (nullable = true)
 |-- action_performed: string (nullable = true)
 |-- bytes_exported: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- employee_role: string (nullable = true)
 |-- session_end_timestamp: string (nullable = true)
 |-- session_start_timestamp: string (nullable = true)
 |-- target_encounter_id: string (nullable = true)
 |-- target_patient_id: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- workstation_ip: string (nullable = true)
 |-- batch_id: string (nullable = false)
 |-- source_file_name: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = false)

['access_log_id', 'action_performed', 'bytes_exported', 'device_id', 'employee_id', 'employee_role', 'session_end_timestamp', 'session_start_timestamp', 'target_encounter_id', 'target_patient_id', 'unexpected_source_field', 'workstation_ip', 'batch_id', 'source_file_name'

In [10]:
# CELL 3 — Write EHR access logs to Bronze Delta table

spark.sql("DROP TABLE IF EXISTS bronze_ehr_access_logs")

(
    ehr_access_logs_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_ehr_access_logs")
)

print(
    "Bronze rows:",
    spark.table("bronze_ehr_access_logs").count()
)

print("bronze_ehr_access_logs created successfully.")

StatementMeta(, 79b6ebbf-f764-456a-a305-652b87a8210c, 12, Finished, Available, Finished, False)

Bronze rows: 1189699
bronze_ehr_access_logs created successfully.


In [11]:
# CELL 4 — Verify Bronze EHR access logs table

bronze_ehr_access_logs_df = spark.table("bronze_ehr_access_logs")

bronze_ehr_access_logs_df.printSchema()
print(bronze_ehr_access_logs_df.columns)
display(bronze_ehr_access_logs_df.limit(20))

StatementMeta(, 79b6ebbf-f764-456a-a305-652b87a8210c, 13, Finished, Available, Finished, False)

root
 |-- access_log_id: string (nullable = true)
 |-- action_performed: string (nullable = true)
 |-- bytes_exported: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- employee_id: string (nullable = true)
 |-- employee_role: string (nullable = true)
 |-- session_end_timestamp: string (nullable = true)
 |-- session_start_timestamp: string (nullable = true)
 |-- target_encounter_id: string (nullable = true)
 |-- target_patient_id: string (nullable = true)
 |-- unexpected_source_field: string (nullable = true)
 |-- workstation_ip: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)

['access_log_id', 'action_performed', 'bytes_exported', 'device_id', 'employee_id', 'employee_role', 'session_end_timestamp', 'session_start_timestamp', 'target_encounter_id', 'target_patient_id', 'unexpected_source_field', 'workstation_ip', 'batch_id', 'source_file_name', '

SynapseWidget(Synapse.DataFrame, fcb3462a-4b28-400f-bf63-c949ea21a9d2)

In [12]:
# CELL 5 — Validate Bronze EHR access logs ingestion

source_row_count = ehr_access_logs_df.count()
bronze_row_count = bronze_ehr_access_logs_df.count()

print("Source rows:", source_row_count)
print("Bronze rows:", bronze_row_count)

if source_row_count == bronze_row_count:
    print("Validation passed: all EHR access log rows reached Bronze.")
else:
    print("Validation failed: row counts do not match.")

StatementMeta(, 79b6ebbf-f764-456a-a305-652b87a8210c, 14, Finished, Available, Finished, False)

Source rows: 1189699
Bronze rows: 1189699
Validation passed: all EHR access log rows reached Bronze.
